In [1]:
from pathlib import Path

import spikeinterface as si
import spikeinterface.extractors as sie
from spikeinterface_gui import run_mainwindow
from probeinterface import get_probe

In [2]:
# Load the recording and set the probe

recording = sie.read_nwb_recording(
    "../data/raw/sub-KM131_ses-20180116T184757_behavior+ecephys+image.nwb"
)

probe = get_probe(
    manufacturer="cambridgeneurotech",
    probe_name="ASSY-77-H3",
)
probe.wiring_to_device('ASSY-77>Adpt.A64-Om32_2x-sm-NN>RHD2164')

recording= recording.set_probe(probe)

In [3]:
# Load or create sorting analyser from published sorting outputs

ANALYSER_DANDI_FOLDER = Path('../sorting_analysers/dandi_analyser')

if(ANALYSER_DANDI_FOLDER.exists()):
    print('Loading saved dandi analyser...')
    analyser_dandi = si.load_sorting_analyzer(folder='../sorting_analysers/dandi_analyser')
    print('Dandi analyser loaded!')
else:
    sorting = sie.read_nwb_sorting(
    "../data/raw/sub-KM131_ses-20180116T184757_behavior+ecephys+image.nwb"
    )

    analyser_dandi = si.create_sorting_analyzer(
        sorting=sorting,
        recording=recording,
        format='binary_folder',
        folder='../sorting_analysers/dandi_analyser',
        sparse=True
    )

    analyser_dandi.compute("random_spikes", max_spikes_per_unit=500)

    extension_dict = {
    "templates": {},
    "waveforms": {},
    "correlograms": {},
    "noise_levels": {'method': 'std'},
    "spike_amplitudes": {},
    "unit_locations": {},
    "template_similarity": {'method': 'l1'},
    "quality_metrics": {},
    }

    analyser_dandi.compute(extension_dict,n_jobs=8 )

Loading saved dandi analyser...
Dandi analyser loaded!


In [4]:
# Load or create sorting analyser from private sorting outputs

ANALYSER_PRIVATE_FOLDER = Path('../sorting_analysers/private_analyser')

if(ANALYSER_PRIVATE_FOLDER.exists()):
    print('Loading saved private analyser...')
    analyser_private = si.load_sorting_analyzer(folder='../sorting_analysers/private_analyser')
    print('Private analyser loaded!')
else:
    sorting = sie.read_kilosort(
        '../data/processed/sorter_output'
    )

    analyser_private = si.create_sorting_analyzer(
        sorting=sorting,
        recording=recording,
        format='binary_folder',
        folder='../sorting_analysers/private_analyser',
        sparse=True
    )

    analyser_private.compute("random_spikes", max_spikes_per_unit=500)

    extension_dict = {
    "templates": {},
    "waveforms": {},
    "correlograms": {},
    "noise_levels": {'method': 'std'},
    "spike_amplitudes": {},
    "unit_locations": {},
    "template_similarity": {'method': 'l1'},
    "quality_metrics": {},
    }

    analyser_private.compute(extension_dict,n_jobs=8 )

Loading saved private analyser...
Private analyser loaded!


In [5]:
# Load the Dandi analyser into the GUI
run_mainwindow(
    analyser_dandi,
    curation=True
)

In [ ]:
# Load the private analyser into the GUI
run_mainwindow(
    analyser_private,
    curation=True
)